# Data simulation and feature engineering

Purpose

Generate synthetic spatial omics → build ExerkineMap graph → compute diffusion → construct ML-ready features.

This notebook replaces all early-stage prototyping.

# Setup & Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csgraph
from scipy.linalg import expm

np.random.seed(42)

# Synthetic Spatial Omics Generator

def generate_synthetic_spatial(
    n_cells=1500,
    n_ligands=40,
    n_receptors=35,
    n_cell_types=6
):
    coords = np.random.uniform(0, 100, (n_cells, 2))
    cell_types = np.random.randint(0, n_cell_types, n_cells)

    ligands = np.random.gamma(2, 1, (n_cells, n_ligands))
    receptors = np.random.gamma(2, 1, (n_cells, n_receptors))

    for ct in range(n_cell_types):
        idx = cell_types == ct
        ligands[idx] += np.random.normal(ct, 0.4, (idx.sum(), n_ligands))

    return coords, ligands, receptors, cell_types

coords, lig, rec, cell_types = generate_synthetic_spatial()

## Spatial Graph Construction

In [ ]:
def generate_synthetic_spatial(
    n_cells=1500,
    n_ligands=40,
    n_receptors=35,
    n_cell_types=6
):
    coords = np.random.uniform(0, 100, (n_cells, 2))
    cell_types = np.random.randint(0, n_cell_types, n_cells)

    ligands = np.random.gamma(2, 1, (n_cells, n_ligands))
    receptors = np.random.gamma(2, 1, (n_cells, n_receptors))

    for ct in range(n_cell_types):
        idx = cell_types == ct
        ligands[idx] += np.random.normal(ct, 0.4, (idx.sum(), n_ligands))

    return coords, ligands, receptors, cell_types

coords, lig, rec, cell_types = generate_synthetic_spatial()

## Ligand-Receptor Scoring

In [ ]:
for i, j in G.edges():
    score = np.dot(lig[i], rec[j])
    G.edges[i, j]["lri_score"] = score

## Signal Diffusion

In [ ]:
A = nx.to_numpy_array(G, weight="lri_score")
L = csgraph.laplacian(A, normed=True)

f0 = lig.mean(axis=1)
F = expm(-0.5 * L) @ f0

## Feature Engineering

In [ ]:
features = []

for node in G.nodes():
    neigh = list(G.neighbors(node))
    if not neigh:
        continue

    scores = [G.edges[node, n]["lri_score"] for n in neigh]

    row = {
        "mean_lri": np.mean(scores),
        "max_lri": np.max(scores),
        "entropy_lri": -np.sum(
            (np.array(scores)/np.sum(scores))
            * np.log(np.array(scores)/np.sum(scores)+1e-8)
        ),
        "diffusion_signal": F[node],
        "cell_type": cell_types[node]
    }

    features.append(row)

X = pd.DataFrame(features)

## Simulated Clincial Outcome

In [ ]:
y = (
    X["mean_lri"]
    + 0.5 * X["diffusion_signal"]
    + np.random.normal(0, 0.3, len(X))
) > X["mean_lri"].median()

X["response"] = y.astype(int)
X.to_parquet("data/processed/synthetic_exerkinemap.parquet")